# Phase 4 — Qwen3 SFT dataset construction

Builds balanced, model-specific prompt-completion shards from the completed Phase 3 artifacts. Run with Internet enabled and Accelerator set to None.

In [ ]:
import platform
import subprocess
import sys
import tarfile
from pathlib import Path

BRANCH = "refactoring/phase-transformer"
REPOSITORY = "https://github.com/SachinGupta2012/Text-Summarizer.git"
PROJECT_ROOT = Path("/kaggle/temp/text-summarizer/project")
OUTPUT_DIRECTORY = Path("/kaggle/working/text-summarizer/phase4/qwen3-sft-v1")

print(f"Python: {platform.python_version()}")
print(f"Platform: {platform.platform()}")
print(f"Output: {OUTPUT_DIRECTORY}")

In [ ]:
if not (PROJECT_ROOT / ".git").is_dir():
    PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            BRANCH,
            "--single-branch",
            REPOSITORY,
            str(PROJECT_ROOT),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "checkout", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )

commit = subprocess.check_output(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
ml_source = str(PROJECT_ROOT / "ml" / "src")
if ml_source not in sys.path:
    sys.path.insert(0, ml_source)
print(f"Project commit: {commit}")

In [ ]:
working_manifest = Path(
    "/kaggle/working/text-summarizer/phase3/full-20261114T114813Z/final/"
    "training-ready/phase3-manifest.json"
)
input_root = Path("/kaggle/input")
input_manifests = sorted(input_root.rglob("phase3-manifest.json"))
manifest_candidates = [working_manifest] if working_manifest.is_file() else input_manifests
if not manifest_candidates:
    archives = sorted(input_root.rglob("phase3-for-phase4.tar.gz"))
    if len(archives) != 1:
        raise RuntimeError(f"Expected one Phase 3 handoff archive; found {archives}")
    extraction_root = Path("/kaggle/temp/text-summarizer/phase3-input")
    extracted_manifest = extraction_root / "phase3-handoff/training-ready/phase3-manifest.json"
    if not extracted_manifest.is_file():
        extraction_root.mkdir(parents=True, exist_ok=True)
        with tarfile.open(archives[0], mode="r:gz") as archive:
            archive.extractall(extraction_root, filter="data")
    manifest_candidates = [extracted_manifest]
manifest_candidates = [path for path in manifest_candidates if path.parent.name == "training-ready"]
if len(manifest_candidates) != 1:
    raise RuntimeError(
        f"Expected exactly one Phase 3 training-ready input; found {manifest_candidates}"
    )
TRAINING_READY_DIRECTORY = manifest_candidates[0].parent
print(f"Phase 3 input: {TRAINING_READY_DIRECTORY}")

In [ ]:
from summarizer_ml.training_data import Phase4Config
from transformers import AutoTokenizer

CONFIG = Phase4Config()
tokenizer = AutoTokenizer.from_pretrained(CONFIG.model_id)
probe = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Summarize this document."}],
    tokenize=True,
    add_generation_prompt=True,
)
if not probe:
    raise RuntimeError("Qwen tokenizer chat-template probe returned no tokens.")
print(f"Tokenizer: {CONFIG.model_id}")
print(f"Chat-template probe tokens: {len(probe)}")
print(CONFIG.model_dump_json(indent=2))

In [ ]:
from time import monotonic

from summarizer_ml.training_data import construct_sft_dataset

phase4_started = monotonic()


def report_progress(stage: str, count: int) -> None:
    elapsed_minutes = (monotonic() - phase4_started) / 60
    print(
        f"Phase 4 {stage}: {count:,}; elapsed={elapsed_minutes:.1f}m",
        flush=True,
    )


result = construct_sft_dataset(
    TRAINING_READY_DIRECTORY,
    OUTPUT_DIRECTORY,
    tokenizer,
    CONFIG,
    progress_every=5_000,
    progress_callback=report_progress,
)
print(
    result.manifest.model_dump_json(
        indent=2,
        exclude={"output_artifacts"},
    )
)
print(f"Output artifacts: {len(result.manifest.output_artifacts)}")
print(f"Phase 4 artifacts: {result.output_directory}")
print("Phase 4 Qwen3 SFT dataset gate passed.")